# MolScope Quickstart: End-to-End Conformational Analysis

[![Open in JupyterLab](https://img.shields.io/badge/Launch-JupyterLab-orange.svg)](https://jupyter.org)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

Welcome to **MolScope**! This interactive tutorial walks you through an end-to-end conformational analysis of the canonical fast-folding **HP35 Villin Headpiece** (PDB: `2RJY`).

### What You Will Learn
1. **Initialize a Study**: Set up a stateful study workspace with on-the-fly Periodic Boundary Condition (PBC) imaging.
2. **Multi-Family Featurization**: Extract residue-residue contacts, Cα distances, DSSP secondary structure, and sidechain dihedrals.
3. **Dimensionality Reduction**: Project conformational dynamics onto **TICA** (slow kinetic transitions) and **Contact Kernel PCA** (CKPCA non-linear RKHS embeddings).
4. **State Discovery (DPA)**: Discover metastable conformational free energy basins without arbitrary cluster counts using **Density Peak Advanced**.
5. **Free Energy Landscapes**: Compute and visualize 2D Potential of Mean Force ($\Delta G$ in kcal/mol) surfaces with detected energy minima.
6. **Explainable AI**: Extract transparent, human-readable Boolean decision rules that structurally define each macrostate.
7. **3D Molecular Hotspots**: Map residue importances directly onto PDB B-factors with one-click PyMOL and ChimeraX visualizers.
8. **Interactive Dashboard**: Explore the self-contained offline HTML report (`report.html`).


## 1. Imports and Environment Setup


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import MolScope's high-level interactive Study API
from molscope.api import Study

print(f"Python: {sys.version.split()[0]}")


## 2. Locate System Data and Initialize Study

We use the HP35 Villin Headpiece (596 atoms, 35 residues, 3 $\alpha$-helices forming a hydrophobic core).
We will locate the trajectory (`simulation_run1.xtc`) and topology (`protein.pdb`).


In [ ]:
# Automatically locate tutorial dataset
data_candidates = [
    Path("../mdxplain study/mdxplain/data/2RJY"),
    Path("data/2RJY"),
    Path("../data/2RJY"),
    Path("/Users/nirav/Docs/mdxplain study/mdxplain/data/2RJY"),
]

data_dir = next((p for p in data_candidates if (p / "protein.pdb").is_file()), None)

if data_dir is None:
    raise FileNotFoundError(
        "Could not find 2RJY dataset. Please ensure protein.pdb and simulation_run1.xtc are present."
    )

topology_file = data_dir / "protein.pdb"
trajectory_file = data_dir / "simulation_run1.xtc"

print(f"Topology:   {topology_file}")
print(f"Trajectory: {trajectory_file}")


Now we initialize a stateful `Study` workspace.
The study manages data provenance, cached feature extractions, kinetic coordinates, and export directories.


In [ ]:
out_dir = Path("results/quickstart_2rjy")
out_dir.mkdir(parents=True, exist_ok=True)

study = Study(
    project_id="2rjy_quickstart",
    out_dir=out_dir,
)

# Register the Villin Headpiece system
study.add_system(
    system_id="villin_headpiece",
    topology=topology_file,
    trajectories=[trajectory_file],
    condition_label="wild_type_300K",
)

# Configure Periodic Boundary Condition (PBC) unwrapping and centering
study.configure_imaging(unwrap=True, center_selection="protein", wrap=False)
print("Study initialized successfully.")


## 3. Multi-Family Biophysical Featurization

Unlike tools limited to simple pairwise contacts, MolScope extracts multiple complementary biophysical feature families:
- **`protein.ca_distances_contacts`**: Pairwise Cα distances and binary contact graphs ($\le 8.0$ Å).
- **`protein.secondary_structure_dssp`**: Fractions of $\alpha$-helix, $\beta$-sheet, and coil.
- **`protein.sidechain_torsions`**: Circular $\sin/\cos$ rotamers for key hydrophobic core residues (Phe47, Phe51, Phe58).


In [ ]:
study.configure_features(
    plugins=[
        "protein.ca_distances_contacts",
        "protein.secondary_structure_dssp",
        "protein.sidechain_torsions",
    ],
    plugin_parameters={
        "protein.ca_distances_contacts": {
            "selection": "protein and name CA",
            "contact_cutoff_angstrom": 8.0,
            "max_pairs": 3000,
        },
        "protein.secondary_structure_dssp": {
            "per_residue": False,
        },
        "protein.sidechain_torsions": {
            "residue_ids": [47, 51, 58],  # Hydrophobic core rotamers
        },
    },
    generic_parameters={
        "frame_stride": 5,   # Subsample stride 5 (2,000 frames from 10,001)
        "max_frames": 2000,
    },
)

# Extract features
print("Extracting multi-family biophysical features...")
study.run(extract_only=True)

features = study.get_features()
manifest = study.get_manifest()

print(f"Extracted {features.shape[1]} features across {features.shape[0]} trajectory frames.")
features.iloc[:5, :6]


## 4. Dimensionality Reduction: Variational TICA & Contact KPCA

To understand conformational transitions, we project high-dimensional coordinates onto:
1. **TICA (Time-lagged Independent Component Analysis)**: Finds the slowest kinetic relaxation modes by solving the generalized eigenvalue problem $C_\tau v_i = \lambda_i C_0 v_i$.
2. **Contact Kernel PCA (CKPCA)**: Non-linear kernel projection ($k(x, y) = \exp(-\gamma \|x-y\|^2)$) capturing non-linear contact manifolds.


In [ ]:
# 1. Fit Variational TICA (lag time tau = 2 frames = 1.0 ns)
tica_result = study.run_tica(lag_time=2, components=5)

print("TICA Implied Timescales (ps):")
for i, ts in enumerate(tica_result.implied_timescales[:4], 1):
    print(f"  Mode {i}: {ts:.1f} ps")

# 2. Fit Contact Kernel PCA
ckpca_result = study.run_contact_kpca(n_components=4)
print(f"Contact KPCA computed {ckpca_result.coordinates.shape[1]} non-linear components.")


In [ ]:
# Inspect kinetic coordinates
tica_coords = tica_result.coordinates
tica_coords.head()


## 5. Non-Parametric State Discovery: Density Peak Advanced (DPA)

MolScope employs **Density Peak Advanced (DPA)** clustering directly on the TICA kinetic manifold.
DPA automatically finds high-density free energy basins ($\rho$) and saddle-point barriers ($\delta$) **without requiring the user to specify an arbitrary number of clusters ($k$)**.


In [ ]:
# Cluster on the slowest kinetic coordinates (TIC1 vs TIC2)
dpa_result = study.cluster_dpa(coordinate_name="TICA", z_score=2.0)

print(f"Discovered {dpa_result.n_clusters} conformational macrostates.")
for state_id, pop in dpa_result.state_populations.items():
    pct = (pop / len(features)) * 100
    print(f"  State {state_id}: {pop:4d} frames ({pct:5.1f}%)")


In [ ]:
# Plot DPA clusters on the TICA reaction coordinates
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    tica_coords["TIC1"],
    tica_coords["TIC2"],
    c=dpa_result.assignments["Cluster"],
    cmap="tab10",
    s=12,
    alpha=0.7,
)
plt.colorbar(scatter, label="Discovered Macrostate")
plt.xlabel("TIC 1 (Slowest Kinetic Coordinate)")
plt.ylabel("TIC 2")
plt.title("MolScope DPA Conformational State Discovery on TICA Manifold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


## 6. 2D Free Energy Surface (FES) Landscape

We compute the potential of mean force:
$$\Delta G(x, y) = -k_B T \ln P(x, y)$$
and identify local thermodynamic energy basins ($B_1, B_2, \dots$).


In [ ]:
fes_out = study.plot_free_energy_landscape(
    dim1="TIC1",
    dim2="TIC2",
    temperature_k=300.0,
    units="kcal/mol",
    bins=50,
    smooth=1.0,
)

print(f"Static FES plot:      {fes_out['static_plot']}")
print(f"Interactive HTML FES: {fes_out['interactive_plot']}")
print(f"Detected {len(fes_out['basins'])} free energy basins:")
fes_out['basins']


## 7. Explainable AI: Decision Tree Boolean Rule Extraction

Rather than treating conformational states as black boxes, MolScope extracts human-interpretable Boolean decision rules:
$$\text{IF } [\text{Feature}_A \le \theta_A] \text{ AND } [\text{Feature}_B > \theta_B] \implies \text{Macrostate } S_k$$


In [ ]:
explainer = study.explain_clusters(
    clustering_name="dpa",
    max_depth=3,
    min_samples_leaf=10,
)

print("Extracted Boolean Biophysical Rules for Discovered States:")
for rule in explainer.rules:
    print(f"\n--- State {rule.cluster_id} ({rule.cluster_name}) ---")
    print(f"  Precision: {rule.precision * 100:.1f}% | Recall: {rule.recall * 100:.1f}% | Support: {rule.support_fraction * 100:.1f}%")
    print(f"  Rule: {rule.simplified_rule_text()}")


## 8. 3D Hotspot Mapping for PyMOL & ChimeraX

MolScope writes residue-level ML importance scores directly into the crystallographic B-factor field of a prepared PDB, allowing instant 3D visualization.


In [ ]:
vis_exports = study.export_3d(
    score_column="TICA_Salience",
    top_labels=6,
)

print("Generated 3D Visualization Assets:")
for name, path in vis_exports.items():
    print(f"  {name:15s}: {path}")


To visualize in PyMOL:
```bash
pymol results/quickstart_2rjy/3d_visualization/2rjy_quickstart_importance.pml
```
Residues critical for state transitions will be highlighted with a red-white-blue heat spectrum.


## 9. Conclusion and Next Steps

In this quickstart tutorial, we used MolScope to:
1. Ingest raw MD trajectories with zero-leakage PBC imaging.
2. Extract multi-family biophysical descriptors (contacts, secondary structure, dihedrals).
3. Compute variational TICA and non-linear Contact KPCA manifolds.
4. Discover free energy basins using non-parametric Density Peak Advanced (DPA) clustering.
5. Compute the 2D Free Energy Surface ($\Delta G$ kcal/mol) with basin detection.
6. Extract transparent Boolean rules explaining the physical determinants of each state.
7. Export 3D PyMOL and ChimeraX structural visualizations.

**Next Tutorials:**
- `02_Simulation_to_Discovery_Pipeline.ipynb`: Running automated OpenMM simulations directly from PDB files.
- `03_Statistical_ML_and_SAM_MFP_Rigor.ipynb`: Machine Learning rigor, purged cross-validation, and solving the Variance Fallacy.
- `04_Multi_Family_Biophysical_Featurization.ipynb`: Deep dive into 12+ biophysical feature families.
